"Objective: Predict whether a patient has heart disease using UCI dataset, Logistic Regression model"


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("heart_disease_uci.csv")

In [ ]:
data.head(15)

In [ ]:
data.isnull().sum()

In [ ]:
data.info()

In [ ]:
#droping columns "ca,thal,slope" 
# becuase they have too mcuh missing values and filling with fake data can affect the model
data = data.drop(columns=['ca','thal','slope'])

In [ ]:
data.isnull().sum()

In [ ]:
 #Filling the missing values
data['trestbps'] = data['trestbps'].fillna(data['trestbps'].median())
data['chol'] = data['chol'].fillna(data['chol'].median())
data['fbs'] = data['fbs'].fillna(data['fbs'].mode()[0])
data['restecg'] = data['restecg'].fillna(data['restecg'].mode()[0])
data['thalch'] = data['thalch'].fillna(data['thalch'].median())
data['exang'] = data['exang'].fillna(data['exang'].mode()[0])
data['oldpeak'] = data['oldpeak'].fillna(data['oldpeak'].median())


In [ ]:
data = data.drop(columns=['id','dataset'])

In [ ]:
data.dtypes

In [ ]:
data['num'] = (data['num']>0).astype(int)

In [ ]:
data.select_dtypes(include='object').columns

In [ ]:
#converting string values to int
data = pd.get_dummies(data,columns=['sex','cp','restecg'],dtype=int)

In [ ]:
data.head()

In [ ]:
#EDA
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
sns.countplot(x='num',data=data)

In [ ]:
plt.hist(data['age'],bins=20)

In [ ]:

sns.boxplot(x='num',y='age',data=data)

In [ ]:
sns.boxplot(x='num',y='chol',data=data)
#chol contains outliers possibly indicating data entry errors

In [ ]:
sns.boxplot(x='num',y='thalch',data=data)

In [ ]:
sns.heatmap(data.corr(),annot=True)

In [ ]:
print(data.corr()['num'].sort_values())


In [ ]:
# EDA Conclusions:
#  cp_asymptomatic, exang, thalch, oldpeak are strongest predictors
# chol is weak predictor despite common belief  
# females less likely to have disease in this dataset
#  age moderately predicts disease  

In [ ]:
#model training
from sklearn.model_selection import train_test_split

X = data.drop(['num'],axis=1)
Y = data['num']

X_train,X_test,Y_train,Y_test = train_test_split(X,Y, test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train,Y_train)

In [ ]:
#Predictiing
pred1 = lr.predict(X_test)
print(pred1)

In [ ]:
#accuracy
from sklearn.metrics import accuracy_score
print(accuracy_score(Y_test,pred1))

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test,pred1)
print(cm)



In [ ]:
#from confuion matrix we can coclude that
# 59 poeple were predicted to be fine were actually fine....
# 16 were called sick but were actually fine
# 21 were called fine but were actually sick
# 88 were called sick and actually were sick

In [ ]:
#ROC
from sklearn.metrics import  roc_auc_score,roc_curve

y_prob = lr.predict_proba(X_test)[:,1]
fpr,tpr,threshold = roc_curve(Y_test,y_prob)
auc = roc_auc_score(Y_test,y_prob)

plt.plot(fpr,tpr)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve - AUC: {auc:.2f}')
plt.show()

In [ ]:
# model correctly distinguishes sick from healthy patients 89% of the time regardless of threshold.

In [ ]:
#Important featurws affecting predictions
importance = pd.Series(lr.coef_[0], index = X.columns)
importance.sort_values().plot(kind='barh')
plt.show()